# DX702 Coding Quiz: Week 9

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy.stats import t,skew
from tqdm import tqdm
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors

# from sklearn.linear_model import LinearRegression

In [ ]:
def simulate(A = 1, B = 1, C = 10, D = 1000):
    """
    Generates synthetic data using a simple linear model with Gaussian noise.

    Parameters
    ----------
    A : float, optional (default=1)
        Coefficient applied to the intermediate variable X in the final output Y.
    B : float, optional (default=1)
        Standard deviation of the noise added to W to generate X.
    C : float, optional (default=10)
        Standard deviation of the noise added to the final output Y.
    D : int, optional (default=1000)
        Number of samples to generate.

    Returns
    -------
    W : ndarray
        Base noise sampled from a standard normal distribution.
    X : ndarray
        Intermediate variable generated by adding Gaussian noise to W.
    Y : ndarray
        Final output generated by applying a linear transformation to X,
        subtracting W, and adding additional Gaussian noise.

    Description
    -----------
    This function simulates a simple generative process:
      - W is sampled from a standard normal distribution.
      - X is generated by adding Gaussian noise (std=B) to W.
      - Y is computed as A * X - W plus additional Gaussian noise (std=C).
    Useful for testing models and understanding noise effects in synthetic data.
    """
    W = np.random.normal(0, 1, D)
    X = W + np.random.normal(0, B, D)
    Y = A * X - W + np.random.normal(0, C, D)
    return W, X, Y

### Question 1  
**10 Points**  

Which of the following is closest to the probability of detecting a nonzero effect of ﻿X﻿ on ﻿Y﻿ (the t-value of ﻿X﻿ is greater in absolute value than about 1.96) given A = 1, B = 1, C = 10, D = 1000? Include W in the regression.


- Option A: 98%  
- Option B: 83%  
- Option C: 93% 
- Option D: 88%

In [3]:
W, X, Y = simulate()

df = pd.DataFrame({'W': W, 'X': X, 'Y': Y})
df

,W,X,Y
0,-0.246191,-0.096546,-6.536937
1,0.275654,0.175372,-7.254305
2,-1.111036,0.692726,7.265652
3,0.718141,1.153805,-2.657460
4,0.555025,-1.510071,-15.683240
...,...,...,...
995,0.296873,1.734759,13.125472
996,-0.892227,-0.325042,12.961390
997,-1.236445,-1.808193,-2.072062
998,-0.397993,-0.449407,8.562491


In [4]:
num_simulations = 1000
significant_count = 0
coef_X = [] 

for _ in range(num_simulations):
    W, X, Y = simulate()
    df = pd.DataFrame({'W': W, 'X': X, 'Y': Y})

    X_reg = sm.add_constant(df[['X', 'W']])
    model = sm.OLS(df['Y'], X_reg).fit()

    t_value = model.tvalues['X']
    if abs(t_value) > 1.96:
        significant_count += 1

    coef_X.append(model.params['X'])

skewness = skew(coef_X)
print(f"Skewness of the coefficient estimates for X: {skewness:.3f}")

probability = significant_count / num_simulations
print(f"Probability of detecting a nonzero effect of X on Y: {probability:.3f}")

Skewness of the coefficient estimates for X: -0.036
Probability of detecting a nonzero effect of X on Y: 0.885


### Question 2  
**10 Points**  

Which of the following is closest to the skew of the estimate in that case? (You can compute this using scipy.)

- Option A: 0.15 
- Option B: -0.15 
- Option C: 0.34 
- Option D: 0

In [5]:
skewness = skew(coef_X)


### Question 3  
**10 Points**  

With A = 1, C = 10, D = 1,000, what value of B is needed to detect that the Data Generating Process (DGP) has a nonzero coefficient for X about 50% of the time? (Choose the closest value.)

Option A
1.8

Option B
0.6

Option C
5.4

Option D
0.2

In [6]:


# Simulation parameters
A = 1
C = 10
D = 1000
target_power = 0.5
t_threshold = 1.96
num_simulations = 500

def simulate_and_test(B):
    detections = 0
    for _ in range(num_simulations):
        W = np.random.normal(0, 1, D)
        X = W + np.random.normal(0, B, D)
        Y = A * X - W + np.random.normal(0, C, D)
        XW = sm.add_constant(np.column_stack((X, W)))
        model = sm.OLS(Y, XW).fit()
        t_val = model.tvalues[1]  # t-value for X
        if abs(t_val) > t_threshold:
            detections += 1
    return detections / num_simulations

# Binary search for B
low, high = 0.01, 5
tolerance = 0.01
best_B = None

while high - low > tolerance:
    mid = (low + high) / 2
    power = simulate_and_test(mid)
    if power < target_power:
        low = mid
    else:
        high = mid
    best_B = mid

print(f"Estimated value of B for ~50% detection rate: {best_B}")



Estimated value of B for ~50% detection rate: 0.6434960937499998


### Question 4  
**10 Points**  

With B = 1, C = 10, D = 100 (note the different value of D), what value of A is needed to detect that the DGP has a nonzero coefficient for X about 50% of the time? (Choose the closest value.) 

Option A
0.5

Option B
1.0

Option C
2.0

Option D
4.0

In [7]:
import numpy as np
import statsmodels.api as sm

# Simulation parameters
B = 1
C = 10
D = 100
target_power = 0.5
t_threshold = 1.96
tolerance = 0.01
max_iterations = 30
simulations_per_A = 1000

def simulate_and_test(A, B, C, D):
    significant_count = 0
    for _ in range(simulations_per_A):
        W = np.random.normal(0, 1, D)
        X = W + np.random.normal(0, B, D)
        Y = A * X - W + np.random.normal(0, C, D)
        XW = sm.add_constant(np.column_stack((X, W)))
        model = sm.OLS(Y, XW).fit()
        t_val = model.tvalues[1]  # t-value for X
        if abs(t_val) > t_threshold:
            significant_count += 1
    return significant_count / simulations_per_A

# Binary search for A
low, high = 0, 2
best_A = None

for _ in range(max_iterations):
    mid = (low + high) / 2
    power = simulate_and_test(mid, B, C, D)
    if abs(power - target_power) < tolerance:
        best_A = mid
        break
    elif power < target_power:
        low = mid
    else:
        high = mid

# If not found within tolerance, return closest estimate
if best_A is None:
    best_A = (low + high) / 2

print(f"Estimated value of A for 50% detection rate: {best_A:.4f}")

Estimated value of A for 50% detection rate: 1.9688
